# Capacity Utilization & Throttling Ingestion (Fabric Capacity Metrics app)

This notebook pulls **CU % consumption** and **throttling** signals for each capacity out
of the **Fabric Capacity Metrics** app (the Microsoft template app that must already be
installed in your tenant), and lands them into a Lakehouse Delta table so they can be
joined against `cost_fabric_api` (from the ingestion notebook next to this one) to answer:
*"where are we consuming more CUs, and is that why cost is going up?"*

It queries the app's semantic model with **DAX** via
[`sempy.fabric.evaluate_dax`](https://learn.microsoft.com/en-us/python/api/semantic-link-sempy/sempy.fabric.evaluate_dax) —
the same technique used by the
[**FUAM**](../../../fabric-unified-admin-monitoring) solution accelerator's
`01_Transfer_CapacityMetricData_Timepoints_Unit` notebook, simplified here to the columns
needed for chargeback/throttling analysis. `sempy` ships with the Fabric Spark runtime, so
no extra install is needed, and auth is handled automatically for the identity running the
notebook — the same "your own admin ID" pattern as the cost notebook.

**Required access**: the identity running this notebook needs at least **Viewer** on the
workspace that hosts the Fabric Capacity Metrics app's semantic model.

> ℹ️ The Capacity Metrics app's semantic model schema has changed across app versions
> (column names like `Capacity Id` vs `capacity Id` differ). This sample targets recent
> versions with a single query shape. If it errors on your tenant, reuse FUAM's
> `01_Transfer_CapacityMetricData_Timepoints_Unit.Notebook`, which tries multiple schema
> versions in sequence — it's a drop-in replacement for the DAX-query cell below.

## Step 0 – Parameters

In [ ]:
import sempy.fabric as fabric
from notebookutils import mssparkutils
from datetime import datetime, timedelta
import pandas as pd
from pyspark.sql.functions import col, lit, current_timestamp

# Workspace and semantic-model (dataset) IDs of the installed Fabric Capacity Metrics app.
# Find them by opening the app's "Fabric Capacity Metrics" report, then Settings/About,
# or via the Fabric REST API (list workspaces you can access, find the one named
# "Fabric Capacity Metrics App" and its semantic model item id).
metric_workspace = "<metrics-app-workspace-id>"
metric_dataset = "<metrics-app-semantic-model-id>"

# How many trailing days to (re)load on each run
days_in_scope = 3

target_table = "capacity_utilization"

## Step 1 – Identify capacities exposed by the app

Same capacities that show up in the Fabric Capacity Metrics report.

In [ ]:
capacity_query = """
EVALUATE SELECTCOLUMNS(Capacities, "CapacityId", Capacities[Capacity Id], "State", Capacities[state])
"""

try:
    capacities_df = fabric.evaluate_dax(workspace=metric_workspace, dataset=metric_dataset, dax_string=capacity_query)
except Exception:
    # Older app versions use lowercase "capacity Id"
    capacity_query = """
    EVALUATE SELECTCOLUMNS(Capacities, "CapacityId", Capacities[capacity Id], "State", Capacities[state])
    """
    capacities_df = fabric.evaluate_dax(workspace=metric_workspace, dataset=metric_dataset, dax_string=capacity_query)

capacities_df.columns = ["CapacityId", "State"]
print(f"Found {len(capacities_df)} capacity(ies) in the Metrics app")
display(capacities_df)

## Step 2 – Pull CU % and throttling metrics per capacity, per day

Key measures pulled (all from the app's `All Measures` table):
- **Background/Interactive billable CU %** — the two components that make up total CU consumption
- **CU limit** and **Cumulative CU usage % (preview)** — how close to the capacity's ceiling
- **Dynamic interactive/background rejection %** — the direct **throttling** signal: > 0 means
  operations were being rejected/delayed because the capacity was over its limit
- **Carry-over add/burndown/cumulative %** — smoothing debt building up or paying down

In [ ]:
def build_dax_query(capacity_id, year, month, day):
    return f"""
    DEFINE
      MPARAMETER 'CapacitiesList' = {{ "{capacity_id}" }}
      VAR __DS0Core =
        SUMMARIZECOLUMNS(
          Capacities[Capacity Id],
          'TimePoints'[TimePoint],
          TREATAS({{"{capacity_id}"}}, 'Capacities'[Capacity Id]),
          TREATAS({{DATE({year}, {month}, {day})}}, 'Dates'[Date]),
          "BackgroundPct", 'All Measures'[Background billable CU %],
          "InteractivePct", 'All Measures'[Interactive billable CU %],
          "CULimit", 'All Measures'[CU limit],
          "CumulativeCUUsagePct", 'All Measures'[Cumulative CU usage % preview],
          "InteractiveRejectionPct", 'All Measures'[Dynamic interactive rejection %],
          "InteractiveRejectionThreshold", 'All Measures'[Interactive rejection threshold],
          "BackgroundRejectionPct", 'All Measures'[Dynamic background rejection %],
          "BackgroundRejectionThreshold", 'All Measures'[Background rejection threshold],
          "CarryOverAddedPct", 'All Measures'[Carry over add %],
          "CarryOverBurndownPct", 'All Measures'[Carry over burndown %],
          "CarryOverCumulativePct", 'All Measures'[Carry over cumulative %]
        )
    EVALUATE __DS0Core
    """

def iterate_dates(n_days):
    today = datetime.now().date()
    return [today - timedelta(days=i) for i in range(n_days, -1, -1)]

frames = []
for cap in capacities_df["CapacityId"]:
    for d in iterate_dates(days_in_scope):
        try:
            q = build_dax_query(cap, d.year, d.month, d.day)
            day_df = fabric.evaluate_dax(workspace=metric_workspace, dataset=metric_dataset, dax_string=q)
            if not day_df.empty:
                frames.append(day_df)
        except Exception as e:
            print(f"WARN: failed for capacity {cap} on {d}: {e}")

raw_df = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
print(f"Retrieved {len(raw_df)} capacity/timepoint rows")

## Step 3 – Derive throttling flags and land into a Lakehouse Delta table

In [ ]:
if not raw_df.empty:
    raw_df.columns = [c.replace("[", "").replace("]", "").split("'")[-1] if "[" in c else c for c in raw_df.columns]
    df = spark.createDataFrame(raw_df)

    df = (
        df
        .withColumn("IsThrottled", (col("InteractiveRejectionPct") > 0) | (col("BackgroundRejectionPct") > 0))
        .withColumn("IsOverCULimit", col("CumulativeCUUsagePct") >= 100)
        .withColumn("IngestedAt", current_timestamp())
    )

    if spark.catalog.tableExists(target_table):
        min_tp = raw_df["TimePoint"].min()
        spark.sql(f"DELETE FROM {target_table} WHERE TimePoint >= '{min_tp}'")
        df.write.mode("append").option("mergeSchema", "true").format("delta").saveAsTable(target_table)
    else:
        df.write.mode("overwrite").format("delta").saveAsTable(target_table)

    throttled_days = df.filter(col("IsThrottled")).select("CapacityId").distinct().count()
    print(f"Wrote {df.count()} rows to '{target_table}' — {throttled_days} capacity/day(s) show throttling in this window")
else:
    print("No rows returned — check metric_workspace/metric_dataset and capacity scope.")

## Next steps

- Join `capacity_utilization` to `cost_fabric_api` on `CapacityId` (map via `dim_fabric_capacities`'s
  resource id) and `TimePoint`/`UsageDate` — spikes in `Cost` should line up with days where
  `IsThrottled = true` or `CumulativeCUUsagePct` approaches 100, confirming *why* cost went up.
- For **workspace/item-level chargeback** (not just capacity-level), extend this notebook using
  the Metrics app's `Items` table (per-item CU consumption) to compute each workspace's **share**
  of a capacity's total CU, then allocate that capacity's dollar cost proportionally — see the
  chargeback formula in [`../../samples/notebook-api-ingestion/README.md`](./README.md).